# Candlestick + Moving average

---

## Importing libraries

In [16]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as plotly # Has better plot for candlesticks

---

# Importing data and doing basic cleaning operations

- Importing multiple tickers using the `download` function from yfinance for a period of 10 years `auto_adjust=True` will also automatically adjust the prices taking into consideration dividends and stock splits.
- Dropping rows with nan or 0 values as they are non trading days and will cause the data to be slightly skewed.
- Changing the data type of volume to float64 for consistency.

In [17]:
# Just copy the code from 1_EDA_Cleaning
tickers = ['HDFCBANK.NS', 'ICICIBANK.NS', '^NSEBANK']
data = yf.download(tickers, period = "5y", auto_adjust=True)
data_dropped = data[~(data == 0).any(axis=1)].copy()
data_dropped = data_dropped[~data_dropped.isna().any(axis=1)].copy()
data_cleaned = data_dropped.copy()
data_cleaned['Volume'] = data_cleaned['Volume'].astype('float64').copy()

[*********************100%***********************]  3 of 3 completed


---

## Plotting candlestick and overlaying moving averages

- Using plotly library to plot the candlestick chart as it gives an interactive viz that can be easily navigated.
- Removed the slider using `xaxis_rangeslider_visible=False` as using mouse is similarly fast and simple while also gives more screen space to the actual chart.
- Adding scatter points of the moving average and connecting them into a smooth curve for overlaying.

SyntaxError: invalid decimal literal (2077897097.py, line 2)

In [20]:
for ticker in tickers:
    ticker_df = data_cleaned.xs(ticker, axis=1, level=1)

# Using plotly to plot the OHLC data.
    fig = plotly.Figure(data=[plotly.Candlestick(
                    x=ticker_df.index, # As the index is a datetime index
                    open=ticker_df['Open'],
                    high=ticker_df['High'],
                    low=ticker_df['Low'],
                    close=ticker_df['Close'],
                    name=ticker
    )])
    fig.update_layout(xaxis_rangeslider_visible=False, title_text=ticker, margin=dict(l=10, r=10, t=40, b=10))
# Rangleslider = False as other wise it shows a range slider below the viz which is redundant
    ticker_df['MA20'] = ticker_df['Close'].rolling(window=20).mean()
    ticker_df['MA50'] = ticker_df['Close'].rolling(window=50).mean()
# rolling makes it so that the mean does the operation for 20 entries, which are 20 trading days worth of data.

# Adding the moving averages, go to the examples section for individual elements in the documentation.
    fig.add_trace(plotly.Scatter(
    x=ticker_df.index, 
    y=ticker_df['MA20'], 
    mode='lines', 
    name='20 Day MA',
    line=dict(color='blue', width=2)
    ))

    fig.add_trace(plotly.Scatter(
    x=ticker_df.index, 
    y=ticker_df['MA50'], 
    mode='lines', 
    name='50 Day MA',
    line=dict(color='orange', width=2)
    ))

    fig.show()
    fig.write_image(f"plotly_plot_{ticker}.png", scale=3, width=1000, height=600)

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


---